In [ ]:
import gzip
import io
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path(".").resolve()

SOURCE = PROJECT_ROOT / "7-days-scraping.zip"  # zipped raw snapshots (plain .csv is over the 100 MB GitHub limit)
PREPROCESSED_FILE = Path(SOURCE).parent / f"{Path(SOURCE).name.split('.')[0]}_preprocessed.csv"

# VIRAL_THRESHOLD = 50 
VIRAL_THRESHOLD = 100        # viral if score > Threshold
LABEL_HORIZON_MIN = 720      # Minutes after submission
OBS_WINDOW_MIN = 60          # features use only what is visible in the first N minutes
MAX_FIRST_SEEN_MIN = 30      # post must be first observed by this age 
GAP_FACTOR = 3               # a pause > GAP_FACTOR x normal snapshot interval = collection gap

SPLIT_STRATEGY = "auto"      # "time" (train on older posts, test on newer), "random" (stratified), "auto"
MIN_TEST_POSITIVES = 5       # "auto" uses a time split only if the test set gets at least this many viral posts
TEST_SIZE = 0.2
RANDOM_STATE = 42

REQUIRED_COLS = ["snapshot_time", "id", "title", "url", "domain", "author",
                 "time", "age_min", "score", "comments", "top_rank", "new_rank"]
print("Source:", SOURCE)
print("Output:", PREPROCESSED_FILE)

Source: C:\Users\chahi\Desktop\Lab Networking project\Classification\7-days-scraping.csv
Output: C:\Users\chahi\Desktop\Lab Networking project\Classification\7-days-scraping_preprocessed.csv


## 1. Load & merge snapshots

Reads every `*.csv` / `*.csv.gz` inside the zip or folder (no extraction to disk), skipping macOS metadata, empty files and files with an unexpected header. Overlapping files are fine: duplicates are removed on `(snapshot_time, id)` in step 2.

In [6]:
def _is_snapshot_name(name: str) -> bool:
    n = name.replace("\\", "/")
    base = n.rsplit("/", 1)[-1]
    return (n.lower().endswith((".csv", ".csv.gz"))
            and "__MACOSX/" not in n and not base.startswith("."))

def _read_one(fileobj, name: str):
    try:
        if name.lower().endswith(".gz"):
            fileobj = gzip.GzipFile(fileobj=fileobj)
        part = pd.read_csv(fileobj, dtype=str, keep_default_na=False, na_values=[""])
    except (OSError, EOFError, pd.errors.EmptyDataError, pd.errors.ParserError, UnicodeDecodeError) as e:
        print(f"  ! skipped {name}: {type(e).__name__}: {e}")
        return None
    missing = [c for c in REQUIRED_COLS if c not in part.columns]
    if missing:
        print(f"  ! skipped {name}: missing columns {missing}")
        return None
    return part

def load_snapshots(source) -> pd.DataFrame:
    source = Path(source)
    if not source.exists():
        raise FileNotFoundError(source)
    parts = []
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            names = sorted(n for n in zf.namelist() if _is_snapshot_name(n))
            nested = [n for n in zf.namelist() if n.lower().endswith(".zip")]
            if nested:
                print(f"  ! {len(nested)} nested .zip file(s) ignored — extract them first: {nested[:3]}")
            print(f"Zip: {len(names)} snapshot file(s)")
            for n in names:
                with zf.open(n) as f:
                    parts.append(_read_one(io.BytesIO(f.read()), n))
    elif source.is_dir():
        files = sorted(p for p in source.rglob("*") if p.is_file() and _is_snapshot_name(str(p)))
        print(f"Folder: {len(files)} snapshot file(s)")
        for p in files:
            with open(p, "rb") as f:
                parts.append(_read_one(f, p.name))
    else:
        with open(source, "rb") as f:
            parts.append(_read_one(f, source.name))
    parts = [p for p in parts if p is not None and len(p)]
    if not parts:
        raise ValueError(f"No usable snapshot files found in {source}")
    return pd.concat(parts, ignore_index=True)

raw = load_snapshots(SOURCE)
print("Merged raw shape:", raw.shape)
raw.head(3)

Merged raw shape: (611052, 13)


,snapshot_time,id,type,title,url,domain,author,time,age_min,score,comments,top_rank,new_rank
0,2026-11-07T11:50:49,50396834,story,Replit rebuilt its sandbox engine with microVMs,https://frontier.replit.com/sandbox-engine-mic...,frontier.replit.com,gaplong,2026-11-06T19:06:10,1004.6,89,32,49,500
1,2026-11-07T11:50:49,50396848,story,iPerf,https://iperf.fr/,iperf.fr,gregsadetsky,2026-11-06T19:07:17,1003.5,2,0,NaN,499
2,2026-11-07T11:50:49,50396857,story,"""Does Meta think we can't read?"" Meta abandons...",https://arstechnica.com/tech-policy/2026/11/do...,arstechnica.com,PaulHoule,2026-11-06T19:07:56,1002.9,6,0,NaN,498


## 2. Clean

In [7]:
df = raw.copy()
df["snapshot_time"] = pd.to_datetime(df["snapshot_time"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], errors="coerce")
for col in ["id", "age_min", "score", "comments", "top_rank", "new_rank"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")   # blank rank = not on that page -> NaN

df["domain"] = df["domain"].fillna("(self)")             # blank url/domain = text post
for col in ["url", "title"]:
    df[col] = df[col].fillna("")
df["author"] = df["author"].fillna("(unknown)")

n0 = len(df)
if "type" in df.columns:
    print("Post types:", df["type"].value_counts(dropna=False).to_dict())
    df = df[df["type"].fillna("story") == "story"]
df = df.dropna(subset=["snapshot_time", "time", "id", "age_min", "score", "comments"])
df = df.drop_duplicates(subset=["snapshot_time", "id"])
df[["id", "score", "comments"]] = df[["id", "score", "comments"]].astype("int64")
df = df.sort_values(["id", "snapshot_time"]).reset_index(drop=True)
del raw

print(f"Removed {n0 - len(df):,} rows (non-stories, unparseable, duplicates). Clean: {df.shape}")
print(f"Unique posts: {df['id'].nunique():,}")

Post types: {'story': 610962, 'poll': 90}
Removed 90 rows (non-stories, unparseable, duplicates). Clean: (610962, 13)
Unique posts: 7,521


## 3. Profile the collection

Instead of assuming how the data was collected, measure it: snapshot interval, gaps, how long posts are tracked, and the rank range. The checks below warn when a setting doesn't fit the data.

In [8]:
snap_times = pd.Series(np.sort(df["snapshot_time"].unique()))
intervals = snap_times.diff().dt.total_seconds() / 60
CADENCE_MIN = float(intervals.median())

is_gap = intervals > GAP_FACTOR * CADENCE_MIN
period_id = is_gap.cumsum()
periods = (pd.DataFrame({"t": snap_times, "period": period_id})
           .groupby("period")["t"].agg(start="min", end="max", snapshots="size"))
periods["hours"] = ((periods["end"] - periods["start"]).dt.total_seconds() / 3600).round(1)

TRACK_LIMIT_MIN = float(df["age_min"].max())
NOT_RANKED = int(np.nanmax(df[["top_rank", "new_rank"]].to_numpy())) + 1

print(f"Snapshots:            {len(snap_times)}  ({snap_times.iloc[0]} -> {snap_times.iloc[-1]})")
print(f"Snapshot interval:    median {CADENCE_MIN:.1f} min")
print(f"Collection periods:   {len(periods)}  (gaps found: {int(is_gap.sum())})")
print(f"Posts tracked up to:  {TRACK_LIMIT_MIN:.0f} min old")
print(f"Rank range:           1..{NOT_RANKED - 1}  ('never ranked' encoded as {NOT_RANKED})")
print(f"Days covered:         {sorted(df['time'].dt.day_name().unique())}")

warnings = []
if CADENCE_MIN > MAX_FIRST_SEEN_MIN:
    warnings.append(f"Snapshots every {CADENCE_MIN:.0f} min but MAX_FIRST_SEEN_MIN={MAX_FIRST_SEEN_MIN}: "
                    "many new posts will fail the 'seen from birth' check. Raise MAX_FIRST_SEEN_MIN.")
if OBS_WINDOW_MIN < 2 * CADENCE_MIN:
    warnings.append(f"OBS_WINDOW_MIN={OBS_WINDOW_MIN} covers < 2 snapshots at this interval: "
                    "velocity/acceleration features will be weak.")
if LABEL_HORIZON_MIN > TRACK_LIMIT_MIN:
    raise ValueError(f"LABEL_HORIZON_MIN={LABEL_HORIZON_MIN} exceeds how long posts are tracked "
                     f"({TRACK_LIMIT_MIN:.0f} min). Lower it.")
if periods["hours"].max() * 60 < LABEL_HORIZON_MIN + MAX_FIRST_SEEN_MIN:
    warnings.append("No collection period is long enough for the label horizon: no post will be eligible.")
for w in warnings:
    print("WARNING:", w)
periods

Snapshots:            501  (2026-11-07 11:50:49 -> 2026-11-14 10:20:03)
Snapshot interval:    median 20.0 min
Collection periods:   1  (gaps found: 0)
Posts tracked up to:  1800 min old
Rank range:           1..500  ('never ranked' encoded as 501)
Days covered:         ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


,start,end,snapshots,hours
period,,,,
0,2026-11-07 11:50:49,2026-11-14 10:20:03,501,166.5


## 4. Select posts that can be labelled honestly

A post is **eligible** when:
1. it was first seen at age ≤ `MAX_FIRST_SEEN_MIN` (we saw its early life), and
2. `submission + LABEL_HORIZON_MIN` falls before the end of the collection period it was first seen in (we'd have seen it go viral).

Both depend only on *when* the post was submitted, not on how it performed. Posts that drop out of tracking early (e.g. flagged/deleted) stay in as negatives.

In [9]:
posts = df.groupby("id").agg(
    submit_time=("time", "first"),
    first_snapshot=("snapshot_time", "first"),
    first_seen_age=("age_min", "min"),
    last_seen_age=("age_min", "max"),
    title=("title", "first"),
    domain=("domain", "first"),
    url=("url", "first"),
    author=("author", "first"),
    peak_score_observed=("score", "max"),   # analysis only, never a feature
)

# End of the uninterrupted collection period each post was first seen in
idx = periods["end"].searchsorted(posts["first_snapshot"])
posts["period_end"] = periods["end"].to_numpy()[idx]

posts["seen_from_birth"] = posts["first_seen_age"] <= MAX_FIRST_SEEN_MIN
posts["full_followup"] = posts["submit_time"] + pd.Timedelta(minutes=LABEL_HORIZON_MIN) <= posts["period_end"]
posts["eligible"] = posts["seen_from_birth"] & posts["full_followup"]

print(f"Posts total:                {len(posts):,}")
print(f"  left-censored (excluded): {(~posts['seen_from_birth']).sum():,}")
print(f"  right-censored (excluded):{(posts['seen_from_birth'] & ~posts['full_followup']).sum():,}")
print(f"  eligible:                 {posts['eligible'].sum():,}")

# Tracking check: eligible posts should normally be observed up to the horizon
short = posts["eligible"] & (posts["last_seen_age"] < LABEL_HORIZON_MIN - 2 * CADENCE_MIN)
print(f"  eligible but tracking stopped before the horizon: {short.sum():,} ({short.mean() / max(posts['eligible'].mean(), 1e-9):.1%} of eligible)")
if posts["eligible"].sum() and short.sum() / posts["eligible"].sum() > 0.10:
    print("WARNING: >10% of eligible posts stop being tracked before the horizon — check how the collector "
          "chooses which posts to follow; their labels may be unreliable.")

Posts total:                7,521
  left-censored (excluded): 512
  right-censored (excluded):273
  eligible:                 6,736
  eligible but tracking stopped before the horizon: 51 (0.8% of eligible)


## 5. Define the target: `viral`

`viral = 1` if the post's score exceeded `VIRAL_THRESHOLD` at any snapshot taken at age ≤ `LABEL_HORIZON_MIN`. `viral_at` / `viral_at_age_min` record when it first crossed the threshold.

In [10]:
label_rows = df[df["age_min"] <= LABEL_HORIZON_MIN]
posts["peak_score_horizon"] = label_rows.groupby("id")["score"].max()
posts["viral"] = (posts["peak_score_horizon"] > VIRAL_THRESHOLD).astype(int)

crossed = df[df["score"] > VIRAL_THRESHOLD].groupby("id").agg(
    viral_at=("snapshot_time", "min"), viral_at_age_min=("age_min", "min"))
posts = posts.join(crossed)

print("Age (min) when posts first crossed the threshold (all posts):")
print(posts["viral_at_age_min"].describe().round(0).to_string())
late = (posts["viral_at_age_min"] > LABEL_HORIZON_MIN).sum()
print(f"Posts crossing only after the {LABEL_HORIZON_MIN}-min horizon: {late}  (counted as not viral)")

elig = posts[posts["eligible"]].copy()
print(f"\nEligible posts: {len(elig):,}  |  viral: {elig['viral'].sum()}  ({elig['viral'].mean():.2%})")
if elig["viral"].sum() < 4 or (1 - elig["viral"]).sum() < 4:
    raise ValueError(
        f"Only {elig['viral'].sum()} viral / {(1 - elig['viral']).sum()} non-viral eligible posts — not enough "
        "to train on. Usual causes: collection periods shorter than LABEL_HORIZON_MIN (see step 3), or a "
        "VIRAL_THRESHOLD too high for this data. Try a shorter LABEL_HORIZON_MIN or lower VIRAL_THRESHOLD.")
if elig["viral"].sum() < 30:
    print("NOTE: fewer than 30 viral examples — expect wide error bars on any metric.")

Age (min) when posts first crossed the threshold (all posts):
count     287.0
mean      352.0
std       336.0
min        24.0
25%       126.0
50%       212.0
75%       452.0
max      1779.0
Posts crossing only after the 720-min horizon: 42  (counted as not viral)

Eligible posts: 6,736  |  viral: 222  (3.30%)


## 6. Early-window engagement features (age ≤ `N` minutes)



In [11]:
win = df[df["id"].isin(elig.index) & (df["age_min"] <= OBS_WINDOW_MIN)]
gw = win.groupby("id")

early = gw.agg(
    score_at_N=("score", "last"),
    comments_at_N=("comments", "last"),
    obs_age_at_N=("age_min", "last"),
    best_new_rank=("new_rank", "min"),
    best_top_rank=("top_rank", "min"),
    n_obs_in_window=("score", "size"),
)
half = win[win["age_min"] <= OBS_WINDOW_MIN / 2].groupby("id")["score"].last()
early["score_gain_2nd_half"] = early["score_at_N"] - half.reindex(early.index).fillna(0)

age = early["obs_age_at_N"].clip(lower=1)
early["score_per_min"] = early["score_at_N"] / age
early["comments_per_min"] = early["comments_at_N"] / age
early["comments_per_point"] = early["comments_at_N"] / early["score_at_N"].clip(lower=1)
early["on_front_page_by_N"] = early["best_top_rank"].notna().astype(int)
early[["best_new_rank", "best_top_rank"]] = early[["best_new_rank", "best_top_rank"]].fillna(NOT_RANKED)

no_window = len(elig) - len(early)
print(f"Eligible posts with no snapshot inside the window (dropped): {no_window}")
early.describe().T[["mean", "50%", "max"]].round(2)

Eligible posts with no snapshot inside the window (dropped): 0


,mean,50%,max
score_at_N,3.45,2.00,573.00
comments_at_N,1.01,0.00,415.00
obs_age_at_N,49.93,50.00,60.00
best_new_rank,9.79,8.00,41.00
best_top_rank,464.80,501.00,501.00
n_obs_in_window,2.99,3.00,4.00
score_gain_2nd_half,1.43,0.00,462.00
score_per_min,0.07,0.03,12.04
comments_per_min,0.02,0.00,8.72
comments_per_point,0.17,0.00,7.00


## 7. Static features (known at submission)

- **Title:** length, words, `Show/Ask/Tell HN`, `?`, digit, `[pdf]/[video]` tag, all-caps word.
- **Link:** self/text post, GitHub, YouTube, `.pdf`.
- **Time:** submission hour (cyclic sin/cos); **weekend flag only if the data spans ≥ 14 days** (otherwise it just encodes which day collection happened).
- **Not included yet:** domain and author identity. With enough data they can help, but they must be encoded inside the CV loop (train-only statistics) — add them at the modeling stage.

In [12]:
SPAN_DAYS = (df["time"].max() - df["time"].min()).days

def static_features(p: pd.DataFrame) -> pd.DataFrame:
    t = p["title"]
    out = pd.DataFrame(index=p.index)
    out["title_len"] = t.str.len()
    out["title_words"] = t.str.split().str.len().fillna(0)
    out["is_show_hn"] = t.str.match(r"(?i)^show hn").astype(int)
    out["is_ask_hn"] = t.str.match(r"(?i)^ask hn").astype(int)
    out["is_tell_hn"] = t.str.match(r"(?i)^tell hn").astype(int)
    out["has_question"] = t.str.contains(r"\?").astype(int)
    out["has_number"] = t.str.contains(r"\d").astype(int)
    out["has_media_tag"] = t.str.contains(r"(?i)\[(?:pdf|video|audio)\]").astype(int)
    out["has_caps_word"] = t.str.contains(r"\b[A-Z]{3,}\b").astype(int)

    d = p["domain"].str.lower()
    out["is_self_post"] = (d == "(self)").astype(int)
    out["is_github"] = d.str.contains("github", regex=False).astype(int)
    out["is_youtube"] = d.str.contains(r"youtube|youtu\.be").astype(int)
    out["is_pdf_url"] = p["url"].str.lower().str.endswith(".pdf").astype(int)

    hour = p["submit_time"].dt.hour + p["submit_time"].dt.minute / 60
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    if SPAN_DAYS >= 14:
        out["is_weekend"] = (p["submit_time"].dt.dayofweek >= 5).astype(int)
    return out

static = static_features(elig)
print(f"Data spans {SPAN_DAYS} day(s) -> weekend flag {'included' if SPAN_DAYS >= 14 else 'skipped'}")
static.filter(regex=r"^(is_|has_)").sum()

Data spans 7 day(s) -> weekend flag skipped


is_show_hn        988
is_ask_hn         111
is_tell_hn         23
has_question      451
has_number       1496
has_media_tag     212
has_caps_word    1184
is_self_post      256
is_github         926
is_youtube        196
is_pdf_url         44
dtype: int64

## 8. Assemble the feature matrix

Heavy-tailed counts get `log1p` (stateless, so safe before the split). Scaling is fit on train only.

In [13]:
LOG_FEATURES = ["score_at_N", "comments_at_N", "score_gain_2nd_half",
                "score_per_min", "comments_per_min", "title_len"]

features = early.join(static, how="inner").drop(columns=["obs_age_at_N"])
features[LOG_FEATURES] = np.log1p(features[LOG_FEATURES].clip(lower=0))

X = features.astype(float)
y = elig.loc[X.index, "viral"]

assert not X.isna().any().any(), "NaN in features"
assert np.isfinite(X.to_numpy()).all(), "inf in features"
print("Feature matrix:", X.shape, "| viral:", int(y.sum()))

compare = X.assign(viral=y).groupby("viral").median().T
compare.columns = ["median_non_viral", "median_viral"]
compare.round(3)

Feature matrix: (6736, 25) | viral: 222


,median_non_viral,median_viral
score_at_N,1.099,2.996
comments_at_N,0.000,1.609
best_new_rank,8.000,8.000
best_top_rank,501.000,7.000
n_obs_in_window,3.000,3.000
score_gain_2nd_half,0.000,2.740
score_per_min,0.033,0.342
comments_per_min,0.000,0.071
comments_per_point,0.000,0.200
on_front_page_by_N,0.000,1.000


## 9. Train/test split

- **`time`** — train on older posts, test on the newest `TEST_SIZE` fraction. Mirrors real use (predict future posts) and keeps same-day posts from leaking across sets. Preferred once there is enough data.
- **`random`** — stratified random split. Fallback when the newest slice would hold too few viral posts.
- **`auto`** — `time` if its test set has ≥ `MIN_TEST_POSITIVES` viral posts, else `random`.

In [14]:
order = elig.loc[X.index, "submit_time"].sort_values(kind="stable").index
cut = int(len(order) * (1 - TEST_SIZE))
time_train, time_test = order[:cut], order[cut:]

strategy = SPLIT_STRATEGY
if strategy == "auto":
    strategy = "time" if y.loc[time_test].sum() >= MIN_TEST_POSITIVES else "random"

if strategy == "time":
    X_train, X_test = X.loc[time_train], X.loc[time_test]
    y_train, y_test = y.loc[time_train], y.loc[time_test]
    print(f"Time split at {elig.loc[time_test[0], 'submit_time']}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    print("Stratified random split")

print(f"Train: {X_train.shape}  viral={int(y_train.sum())}")
print(f"Test:  {X_test.shape}  viral={int(y_test.sum())}")
if y_test.sum() < MIN_TEST_POSITIVES:
    print("NOTE: very few viral posts in test — rely on the cross-validation below, not this split.")

Time split at 2026-11-12 17:56:16
Train: (5388, 25)  viral=164
Test:  (1348, 25)  viral=58


## 10. Save



In [16]:
split_col = pd.Series("train", index=X.index).where(X.index.isin(X_train.index), "test")
preprocessed = (elig.loc[X.index, ["title", "submit_time"]]
                .join(X)
                .assign(viral=y, split=split_col)
                .sort_values("submit_time"))
preprocessed.to_csv(PREPROCESSED_FILE, index_label="id")
print(f"Saved {PREPROCESSED_FILE.name}: {preprocessed.shape[0]:,} posts x {preprocessed.shape[1]} columns  "
      f"(viral={int(preprocessed['viral'].sum())}, train={int((split_col == 'train').sum())}, "
      f"test={int((split_col == 'test').sum())})")

Saved 7-days-scraping_preprocessed.csv: 6,736 posts x 29 columns  (viral=222, train=5388, test=1348)
